# GpuIndexRefine Benchmark Notebook

**FP16 / SQ8 / SQ4 refinement storage on GPU — performance testing and interactive charts**

Contents:
1. **Setup** — verify faiss import, load datasets
2. **Live Benchmark** — build indexes, run timed searches, collect results
3. **Interactive Charts** — storage comparison, recall vs VRAM, k_factor sweep, batch scaling

In [61]:
!nvidia-smi

Sat Apr 25 21:58:10 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.183.01             Driver Version: 535.183.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A4000               Off | 00000000:02:00.0 Off |                    0 |
| 41%   48C    P8              14W / 140W |    174MiB / 15352MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [51]:
import sys, os, time, json, warnings
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# Plotly renderer — picks output format that the current Jupyter frontend can show.
#   "iframe"             : writes iframe_figures/*.html and embeds them — works in every frontend
#   "notebook_connected" : inline HTML + plotly.js from CDN — needs trusted notebook
#   "notebook"           : same, plotly.js bundled in the .ipynb (large files)
#   "jupyterlab"         : JupyterLab >= 3 with @jupyter-widgets/jupyterlab-manager
#   "vscode"             : VS Code Jupyter
# Default to "iframe" since it's the most robust; override via PLOTLY_RENDERER env var.
pio.renderers.default = os.environ.get("PLOTLY_RENDERER", "iframe")
print(f"plotly renderer: {pio.renderers.default}")

# --- locate faiss Python module (built in-tree) ---
REPO = os.path.dirname(os.path.abspath("."))
build_libs = sorted(
    [p for p in __import__("glob").glob(f"{REPO}/build/faiss/python/build/lib*/")],
    reverse=True,
)
if build_libs:
    sys.path.insert(0, build_libs[0])
    print(f"faiss path: {build_libs[0]}")
else:
    print("WARNING: faiss build not found — benchmark cells will be skipped")

try:
    import faiss
    print(f"faiss version : {faiss.__version__}")
    print(f"GPU count     : {faiss.get_num_gpus()}")
except ImportError as e:
    print(f"faiss not importable: {e}")
    faiss = None

# colour palette (matches presentation)
COLORS = {"float16": "#1F4E79", "sq8": "#1A5C37", "sq4": "#7B2D2D", "ivfpq": "#555555"}
LABELS = {"float16": "FP16", "sq8": "SQ8", "sq4": "SQ4", "ivfpq": "IVF-PQ"}

plotly renderer: iframe
faiss path: /home/administrator/workspace/faiss/build/faiss/python/build/lib/
faiss version : 1.8.0
GPU count     : 1


## 1  Load Datasets

In [52]:
def read_fvecs(path):
    with open(path, "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.float32)
    d = data[:1].view(np.int32)[0]
    return data.reshape(-1, d + 1)[:, 1:].copy()

def read_ivecs(path):
    with open(path, "rb") as f:
        data = np.frombuffer(f.read(), dtype=np.int32)
    d = data[:1][0]
    return data.reshape(-1, d + 1)[:, 1:].copy()

def read_ibin(path, dtype=np.int32):
    with open(path, "rb") as f:
        nrows, ncols = np.frombuffer(f.read(8), dtype=np.int32)
        data = np.frombuffer(f.read(), dtype=dtype)
    return data.reshape(nrows, ncols).copy()

def read_fbin(path):
    with open(path, "rb") as f:
        nrows, ncols = np.frombuffer(f.read(8), dtype=np.int32)
        data = np.frombuffer(f.read(), dtype=np.float32)
    return data.reshape(nrows, ncols).copy()

TESTS_DIR = os.path.dirname(os.path.abspath("benchmark_notebook.ipynb"))

datasets = {}

# SIFT-1M
sift_dir = os.path.join(TESTS_DIR, "sift")
if os.path.exists(sift_dir):
    xb = read_fvecs(os.path.join(sift_dir, "sift_base.fvecs"))
    xq = read_fvecs(os.path.join(sift_dir, "sift_query.fvecs"))
    gt = read_ivecs(os.path.join(sift_dir, "sift_groundtruth.ivecs"))
    datasets["SIFT1M"] = dict(xb=xb, xq=xq, gt=gt, d=xb.shape[1])
    print(f"SIFT1M  base={xb.shape}  queries={xq.shape}  d={xb.shape[1]}")
else:
    print("SIFT1M not found — skip")

# Wiki-all-1M
wiki_dir = os.path.join(TESTS_DIR, "wiki_all_1M")
if os.path.exists(wiki_dir):
    xb_w = read_fbin(os.path.join(wiki_dir, "base.1M.fbin"))
    xq_w = read_fbin(os.path.join(wiki_dir, "queries.fbin"))
    gt_w = read_ibin(os.path.join(wiki_dir, "groundtruth.1M.neighbors.ibin"))
    datasets["Wiki1M"] = dict(xb=xb_w, xq=xq_w, gt=gt_w, d=xb_w.shape[1])
    print(f"Wiki1M  base={xb_w.shape}  queries={xq_w.shape}  d={xb_w.shape[1]}")
else:
    print("Wiki1M not found — skip")

SIFT1M  base=(1000000, 128)  queries=(10000, 128)  d=128
Wiki1M  base=(1000000, 768)  queries=(10000, 768)  d=768


## 2  Live Benchmark

Build GPU indexes and measure latency + recall in-notebook.  
Configure the parameters below; results are collected into `live_results` and automatically merged into the charts in Section 3.

In [62]:
# ── 参数 ──────────────────────────────────────────────────────────────────────
K             = 10
WARMUP        = 3
RUNS          = 3
NPROBE_SWEEP  = [8, 16, 32, 64, 128]
KFACTOR_SWEEP = [5, 10, 20]
BATCH_SIZES   = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1000]

DS_CFG = {
    "SIFT1M": dict(nlist=512,  m=16, nprobe=32, k_factor=10),
    "Wiki1M":  dict(nlist=1024, m=48, nprobe=32, k_factor=10),
}
VRAM_MB = {
    "float16": lambda nb, d: nb * d * 2 / 1024**2,
    "sq8":     lambda nb, d: nb * d     / 1024**2,
    "sq4":     lambda nb, d: nb * d / 2 / 1024**2,
}
# ─────────────────────────────────────────────────────────────────────────────

def recall_at_k(I, gt, k):
    n = I.shape[0]
    return sum(len(set(I[i, :k]) & set(gt[i, :k])) for i in range(n)) / (n * k)

def build_ivfpq_gpu(res, d, nlist, m, xb_train):
    cfg = faiss.GpuIndexIVFPQConfig()
    cfg.useFloat16LookupTables = True
    cfg.device = 0
    idx = faiss.GpuIndexIVFPQ(res, d, nlist, m, 8, faiss.METRIC_L2, cfg)
    idx.train(xb_train)
    return idx

def build_refine(res, trained_cpu, d, nprobe, xb, storage, k_factor):
    """
    Mirrors test_gpu_refine_sq4.py build pattern (proven working).

    FLOAT16: base.add(xb) first, then flat.add(xb), then GpuIndexRefine(base, flat).
    SQ8/SQ4: GpuIndexRefine(base, cfg) then refine.train(xb) + refine.add(xb).
    Returns (refine_idx, base_idx, refine_flat_or_None).
    Caller keeps all three alive.
    """
    base = faiss.index_cpu_to_gpu(res, 0, trained_cpu)
    base.nprobe = nprobe

    if storage == "float16":
        base.add(xb)                           # populate base first
        flat_cfg = faiss.GpuIndexFlatConfig()
        flat_cfg.useFloat16 = True
        flat = faiss.GpuIndexFlatL2(res, d, flat_cfg)
        flat.add(xb)                           # populate flat
        refine = faiss.GpuIndexRefine(res, base, flat)   # default k_factor=1
        refine.setKFactor(float(k_factor))
        return refine, base, flat

    else:
        cfg = faiss.GpuIndexRefineConfig()
        cfg.storageType = (faiss.RefineStorageType_SQ8 if storage == "sq8"
                           else faiss.RefineStorageType_SQ4)
        cfg.k_factor = float(k_factor)
        refine = faiss.GpuIndexRefine(res, base, cfg)
        refine.train(xb)
        refine.add(xb)
        return refine, base, None

def time_search(idx, xq, k, warmup=WARMUP, runs=RUNS, batch=128):
    """Returns (latency_ms_per_batch, qps)."""
    for _ in range(warmup):
        idx.search(xq[:batch], k)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        for i in range(0, len(xq), batch):
            idx.search(xq[i:i+batch], k)
        times.append(time.perf_counter() - t0)
    t = float(np.mean(times))
    nbatches = len(xq) / batch
    return t / nbatches * 1000, len(xq) / t   # ms/batch, QPS

def time_batch(idx, xq, k, batch_size):
    q = xq[:batch_size]
    for _ in range(WARMUP): idx.search(q, k)
    times = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        idx.search(q, k)
        times.append(time.perf_counter() - t0)
    t = float(np.mean(times))
    return dict(batch_size=batch_size,
                latency_ms=t*1000,
                latency_per_query_us=t/batch_size*1e6,
                qps=batch_size/t)

# ── main ──────────────────────────────────────────────────────────────────────
all_bench = {}   # ds → list of row dicts
all_batch = {}   # ds → {format: [batch_pts]}

res = faiss.StandardGpuResources()
res.setTempMemory(512 * 1024 * 1024)

for ds_name, cfg in DS_CFG.items():
    if ds_name not in datasets:
        print(f"\n{ds_name}: skip"); continue

    ds = datasets[ds_name]
    xb, xq, gt, d = ds["xb"], ds["xq"], ds["gt"], ds["d"]
    nb, nq = xb.shape[0], xq.shape[0]
    nlist, m = cfg["nlist"], cfg["m"]
    nprobe, kf = cfg["nprobe"], cfg["k_factor"]
    train_n = min(nb, nlist * 40)

    print(f"\n{'='*60}\n  {ds_name}  d={d}  nb={nb:,}  nq={nq:,}"
          f"  nlist={nlist}  m={m}\n{'='*60}")

    all_bench[ds_name] = []
    all_batch[ds_name] = {}

    # ── GPU train once → CPU snapshot ──────────────────────────────────────
    print(f"\n  [train] {train_n:,} vectors …", end=" ", flush=True)
    t0 = time.perf_counter()
    base0 = build_ivfpq_gpu(res, d, nlist, m, xb[:train_n])
    trained_cpu = faiss.index_gpu_to_cpu(base0)
    del base0
    print(f"{time.perf_counter()-t0:.1f}s")

    # ── IVF-PQ baseline batch scaling ──────────────────────────────────────
    print(f"  [IVF-PQ baseline] add …", end=" ", flush=True)
    base_bl = faiss.index_cpu_to_gpu(res, 0, trained_cpu)
    base_bl.nprobe = nprobe
    base_bl.add(xb)
    print("done")
    pts = [time_batch(base_bl, xq, K, min(bs, nq)) for bs in BATCH_SIZES]
    all_batch[ds_name]["ivfpq"] = pts
    for p in pts:
        print(f"    bs={p['batch_size']:5d}  {p['latency_per_query_us']:7.1f} µs/q  {p['qps']:8.0f} QPS")
    del base_bl

    # ── storage formats ────────────────────────────────────────────────────
    for storage in ["float16", "sq8", "sq4"]:
        print(f"\n  [{storage.upper()}] build …", end=" ", flush=True)
        t0 = time.perf_counter()
        refine, base, flat = build_refine(res, trained_cpu, d, nprobe, xb, storage, kf)
        print(f"{time.perf_counter()-t0:.1f}s")

        # main result
        _, I = refine.search(xq, K)
        rec = recall_at_k(I, gt, K)
        lat, qps = time_search(refine, xq, K)
        vram = int(VRAM_MB[storage](nb, d))
        print(f"    recall@{K}={rec:.4f}  lat={lat:.2f}ms/batch  QPS={qps:.0f}  VRAM≈{vram}MB")
        all_bench[ds_name].append(dict(
            config=dict(dataset=ds_name, nlist=nlist, m=m,
                        nprobe=nprobe, k=K, k_factor=kf, storage=storage),
            dataset=ds_name, vram_mb=vram, latency_ms=lat, qps=qps, recall=rec))

        # nprobe sweep (float16 only)
        if storage == "float16":
            print("    nprobe sweep …")
            for np_ in NPROBE_SWEEP:
                if np_ == nprobe: continue
                base.nprobe = np_
                _, I = refine.search(xq, K)
                rec_n = recall_at_k(I, gt, K)
                lat_n, qps_n = time_search(refine, xq, K)
                print(f"      nprobe={np_:3d}  recall={rec_n:.4f}  lat={lat_n:.2f}ms/batch")
                all_bench[ds_name].append(dict(
                    config=dict(dataset=ds_name, nlist=nlist, m=m,
                                nprobe=np_, k=K, k_factor=kf, storage=storage),
                    dataset=ds_name, vram_mb=vram, latency_ms=lat_n, qps=qps_n, recall=rec_n))
            base.nprobe = nprobe   # restore

            # k_factor sweep via setKFactor — no rebuild needed
            print("    k_factor sweep …")
            for kf_ in KFACTOR_SWEEP:
                if kf_ == kf: continue
                refine.setKFactor(float(kf_))
                _, I = refine.search(xq, K)
                rec_k = recall_at_k(I, gt, K)
                lat_k, qps_k = time_search(refine, xq, K)
                print(f"      k_factor={kf_:2d}  recall={rec_k:.4f}  lat={lat_k:.2f}ms/batch")
                all_bench[ds_name].append(dict(
                    config=dict(dataset=ds_name, nlist=nlist, m=m,
                                nprobe=nprobe, k=K, k_factor=kf_, storage=storage),
                    dataset=ds_name, vram_mb=vram, latency_ms=lat_k, qps=qps_k, recall=rec_k))
            refine.setKFactor(float(kf))  # restore

        # batch scaling
        print("    batch scaling …")
        pts = [time_batch(refine, xq, K, min(bs, nq)) for bs in BATCH_SIZES]
        all_batch[ds_name][storage] = pts
        for p in pts:
            print(f"      bs={p['batch_size']:5d}  {p['latency_per_query_us']:7.1f} µs/q  {p['qps']:8.0f} QPS")

        del refine, base, flat

del res

# ── save JSON ──────────────────────────────────────────────────────────────────
import os as _os
bench_path = _os.path.join(TESTS_DIR, "benchmark_results.json")
batch_path = _os.path.join(TESTS_DIR, "batch_scaling_results.json")
with open(bench_path, "w") as f: json.dump(all_bench, f, indent=2)
with open(batch_path, "w") as f: json.dump(all_batch, f, indent=2)
print(f"\n✓ {bench_path}")
print(f"✓ {batch_path}")


  SIFT1M  d=128  nb=1,000,000  nq=10,000  nlist=512  m=16

  [train] 20,480 vectors … 0.3s
  [IVF-PQ baseline] add … done
    bs=    1     99.2 µs/q     10083 QPS
    bs=    2     53.3 µs/q     18750 QPS
    bs=    4     37.6 µs/q     26601 QPS
    bs=    8     27.1 µs/q     36874 QPS
    bs=   16     19.4 µs/q     51571 QPS
    bs=   32     13.9 µs/q     72145 QPS
    bs=   64     11.1 µs/q     90442 QPS
    bs=  128     10.2 µs/q     98382 QPS
    bs=  256      8.9 µs/q    112325 QPS
    bs=  512      8.6 µs/q    115928 QPS
    bs= 1000      8.6 µs/q    115730 QPS

  [FLOAT16] build … 0.4s
    recall@10=0.9639  lat=1.58ms/batch  QPS=81149  VRAM≈244MB
    nprobe sweep …
      nprobe=  8  recall=0.8666  lat=0.67ms/batch
      nprobe= 16  recall=0.9367  lat=1.00ms/batch
      nprobe= 64  recall=0.9699  lat=2.58ms/batch
      nprobe=128  recall=0.9705  lat=5.05ms/batch
    k_factor sweep …
      k_factor= 5  recall=0.9048  lat=1.43ms/batch
      k_factor=20  recall=0.9840  lat=1.86ms/ba

## 3  Load Pre-computed Results

Full benchmark JSONs (3-run averages, both datasets, all storage formats).

In [54]:
bench_json  = json.load(open(os.path.join(TESTS_DIR, "benchmark_results.json")))
batch_json  = json.load(open(os.path.join(TESTS_DIR, "batch_scaling_results.json")))

def pick(rows, nprobe, k_factor, storage):
    for r in rows:
        c = r["config"]
        if c["nprobe"] == nprobe and c["k_factor"] == k_factor and c.get("storage") == storage:
            return r
    return None

print("benchmark_results.json  datasets:", list(bench_json.keys()))
print("batch_scaling_results.json datasets:", list(batch_json.keys()))
for ds, rows in bench_json.items():
    storages = sorted({r["config"]["storage"] for r in rows})
    print(f"  {ds}: {len(rows)} rows, storages={storages}")

benchmark_results.json  datasets: ['SIFT1M', 'Wiki1M']
batch_scaling_results.json datasets: ['SIFT1M', 'Wiki1M']
  SIFT1M: 9 rows, storages=['float16', 'sq4', 'sq8']
  Wiki1M: 9 rows, storages=['float16', 'sq4', 'sq8']


## 4  Chart: Storage Format Comparison

Grouped bar chart — Recall@10, Latency, VRAM for each (dataset × storage) combination at nprobe=32, k_factor=10.

In [55]:
# Reload results each time this cell runs
_bench = json.load(open(os.path.join(TESTS_DIR, "benchmark_results.json")))

STORAGES = ["float16", "sq8", "sq4"]
DATASETS = ["SIFT1M", "Wiki1M"]

def pick(rows, nprobe, k_factor, storage):
    for r in rows:
        c = r["config"]
        if (c["nprobe"] == nprobe and c["k_factor"] == k_factor
                and c.get("storage") == storage):
            return r
    return None

# --- build data arrays ---
x_labels = [f"{ds}<br>{LABELS[st]}" for ds in DATASETS for st in STORAGES]
colors    = [COLORS[st]             for ds in DATASETS for st in STORAGES]
recalls, latencies, vrams = [], [], []
for ds in DATASETS:
    for st in STORAGES:
        r = pick(_bench[ds], 32, 10, st)
        recalls.append(round(r["recall"], 4)     if r else 0)
        latencies.append(round(r["latency_ms"], 2) if r else 0)
        vrams.append(r["vram_mb"]                if r else 0)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Recall@10", "Latency (ms / batch=128)", "Refine VRAM (MB)"],
    horizontal_spacing=0.10,
)

for col, (vals, fmt, ymax) in enumerate(
    [(recalls, ".3f", 1.05),
     (latencies, ".2f", None),
     (vrams, ".0f", None)],
    start=1,
):
    fig.add_trace(go.Bar(
        x=x_labels, y=vals,
        marker_color=colors,
        text=[f"{v:{fmt}}" for v in vals],
        textposition="auto",
        showlegend=False,
    ), row=1, col=col)
    if ymax:
        fig.update_yaxes(range=[0, ymax], row=1, col=col)

# legend entries
for st in STORAGES:
    fig.add_trace(go.Bar(
        x=[None], y=[None], name=LABELS[st],
        marker_color=COLORS[st], showlegend=True,
    ), row=1, col=1)

fig.update_layout(
    title_text="Storage Format Comparison — nprobe=32, k_factor=10, k=10",
    height=480, bargap=0.25,
    legend=dict(orientation="h", yanchor="bottom", y=1.04,
                xanchor="center", x=0.5),
    plot_bgcolor="white", paper_bgcolor="white",
)
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0")
fig.show()


## 5  Chart: Recall vs VRAM Trade-off

Each point = one (dataset, storage) configuration. The ideal corner is top-right (high recall) + left (low VRAM).

In [56]:
_bench = json.load(open(os.path.join(TESTS_DIR, "benchmark_results.json")))
fig = go.Figure()

for ds in DATASETS:
    rows = _bench[ds]
    for st in STORAGES:
        r = pick(rows, 32, 10, st)
        if not r:
            continue
        fig.add_trace(go.Scatter(
            x=[r["vram_mb"]], y=[r["recall"]],
            mode="markers+text",
            name=f"{ds} {LABELS[st]}",
            text=[f"{ds} {LABELS[st]}"],
            textposition="top center",
            marker=dict(
                size=18,
                color=COLORS[st],
                symbol="circle" if ds == "Wiki1M" else "square",
                line=dict(color="white", width=2),
            ),
            showlegend=True,
        ))

fig.update_layout(
    title="Recall@10 vs Refine VRAM — nprobe=32, k_factor=10<br>"
          "<sup>Circle = Wiki1M (d=768) · Square = SIFT1M (d=128)</sup>",
    xaxis_title="Refine VRAM (MB)  ←  less is better",
    yaxis_title="Recall@10  ↑  higher is better",
    height=500,
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", y=-0.18),
)
fig.update_xaxes(showgrid=True, gridcolor="#e0e0e0", zeroline=False)
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0", zeroline=False)
fig.show()

## 6  Chart: k_factor Sweep (Wiki1M FP16)

Dual-axis: left = Recall@10 (blue), right = Latency ms (orange).  
Shows the recall/latency knee — k_factor=10 gives the best trade-off.

In [57]:
_bench = json.load(open(os.path.join(TESTS_DIR, "benchmark_results.json")))
kf_rows = [r for r in _bench["Wiki1M"]
           if r["config"]["storage"] == "float16" and r["config"]["nprobe"] == 32]
kf_rows.sort(key=lambda r: r["config"]["k_factor"])

kf_vals  = [r["config"]["k_factor"] for r in kf_rows]
kf_rec   = [r["recall"]     for r in kf_rows]
kf_lat   = [r["latency_ms"] for r in kf_rows]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=kf_vals, y=kf_rec,
    mode="lines+markers",
    name="Recall@10",
    line=dict(color=COLORS["float16"], width=3),
    marker=dict(size=10),
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=kf_vals, y=kf_lat,
    mode="lines+markers",
    name="Latency (ms)",
    line=dict(color="#E67E22", width=3, dash="dot"),
    marker=dict(size=10, symbol="diamond"),
), secondary_y=True)

# annotate the knee
fig.add_vline(x=10, line_dash="dash", line_color="gray", opacity=0.5,
              annotation_text="knee  k_factor=10", annotation_position="top right")

fig.update_layout(
    title="k_factor Sweep — Wiki1M FP16, nprobe=32",
    xaxis=dict(title="k_factor", tickvals=kf_vals),
    height=420,
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", y=1.12),
)
fig.update_yaxes(title_text="Recall@10", secondary_y=False,
                 showgrid=True, gridcolor="#e0e0e0")
fig.update_yaxes(title_text="Latency (ms)", secondary_y=True,
                 showgrid=False)
fig.show()

## 7  Chart: nprobe Sweep (Wiki1M FP16)

Increasing nprobe searches more IVF lists → higher recall but higher latency.  
k_factor=10 held constant.

In [58]:
_bench = json.load(open(os.path.join(TESTS_DIR, "benchmark_results.json")))
np_rows = [r for r in _bench["Wiki1M"]
           if r["config"]["storage"] == "float16" and r["config"]["k_factor"] == 10]
np_rows.sort(key=lambda r: r["config"]["nprobe"])

np_vals = [r["config"]["nprobe"] for r in np_rows]
np_rec  = [r["recall"]     for r in np_rows]
np_lat  = [r["latency_ms"] for r in np_rows]

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=np_vals, y=np_rec,
    mode="lines+markers", name="Recall@10",
    line=dict(color=COLORS["float16"], width=3),
    marker=dict(size=10),
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=np_vals, y=np_lat,
    mode="lines+markers", name="Latency (ms)",
    line=dict(color="#E67E22", width=3, dash="dot"),
    marker=dict(size=10, symbol="diamond"),
), secondary_y=True)

fig.add_vline(x=32, line_dash="dash", line_color="gray", opacity=0.5,
              annotation_text="nprobe=32 (default)", annotation_position="top right")

fig.update_layout(
    title="nprobe Sweep — Wiki1M FP16, k_factor=10",
    xaxis=dict(title="nprobe", tickvals=np_vals),
    height=420,
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", y=1.12),
)
fig.update_yaxes(title_text="Recall@10", secondary_y=False,
                 showgrid=True, gridcolor="#e0e0e0")
fig.update_yaxes(title_text="Latency (ms)", secondary_y=True, showgrid=False)
fig.show()

## 8  Chart: Batch Scaling — Per-query Latency

Log-scale x-axis. Each line = one storage format + IVF-PQ baseline.  
Slope at small batch = launch-overhead bound. Flat at large batch = GEMM-saturated.

In [59]:
_batch = json.load(open(os.path.join(TESTS_DIR, "batch_scaling_results.json")))
SERIES = [("ivfpq", "IVF-PQ (no rerank)"), ("float16", "FP16"),
          ("sq8", "SQ8"), ("sq4", "SQ4")]

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=["Wiki1M (d=768)", "SIFT1M (d=128)"],
                    horizontal_spacing=0.10)

for col, ds in enumerate(["Wiki1M", "SIFT1M"], start=1):
    block = _batch[ds]
    for key, label in SERIES:
        pts = block.get(key, [])
        if not pts:
            continue
        xs = [p["batch_size"] for p in pts]
        ys = [p["latency_per_query_us"] for p in pts]
        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            mode="lines+markers",
            name=label,
            line=dict(color=COLORS.get(key, "#888"), width=2),
            marker=dict(size=7),
            legendgroup=key,
            showlegend=(col == 1),
        ), row=1, col=col)

fig.update_xaxes(type="log", title_text="Batch size", dtick=1)
fig.update_yaxes(title_text="Per-query latency (µs)", row=1, col=1,
                 showgrid=True, gridcolor="#e0e0e0")
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0", row=1, col=2)
fig.update_layout(
    title_text="Batch Scaling — per-query latency (µs) vs batch size",
    height=460,
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
)
fig.show()

## 9  Chart: Batch Scaling — Peak QPS

Peak QPS is achieved at different batch sizes per format.  
All three refine formats land within 2.1% of each other → compute-bound, not bandwidth-bound.

In [60]:
_batch = json.load(open(os.path.join(TESTS_DIR, "batch_scaling_results.json")))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Wiki1M (d=768)", "SIFT1M (d=128)"],
    horizontal_spacing=0.14,
)

for col, ds in enumerate(["Wiki1M", "SIFT1M"], start=1):
    block = _batch[ds]
    for key, label in SERIES:
        pts = block.get(key, [])
        if not pts:
            continue
        xs = [p["batch_size"] for p in pts]
        ys = [p["qps"]        for p in pts]
        peak = max(pts, key=lambda p: p["qps"])

        # bake peak QPS into the legend so the chart itself stays uncluttered
        legend_label = f"{label}  (peak {peak['qps']/1000:.1f}k @ bs={peak['batch_size']})"

        fig.add_trace(go.Scatter(
            x=xs, y=ys,
            mode="lines+markers",
            name=legend_label,
            line=dict(color=COLORS.get(key, "#888"), width=2),
            marker=dict(size=7),
            legendgroup=key,
            showlegend=(col == 1),
        ), row=1, col=col)

        # star at the peak — no text label, hover gives exact value
        fig.add_trace(go.Scatter(
            x=[peak["batch_size"]], y=[peak["qps"]],
            mode="markers",
            marker=dict(size=13, color=COLORS.get(key, "#888"),
                        line=dict(color="white", width=2), symbol="star"),
            legendgroup=key, showlegend=False,
            hovertemplate=f"{label} peak<br>bs=%{{x}}<br>QPS=%{{y:,.0f}}<extra></extra>",
        ), row=1, col=col)

fig.update_xaxes(type="log", title_text="Batch size", dtick=1)
fig.update_yaxes(title_text="QPS", row=1, col=1,
                 showgrid=True, gridcolor="#e0e0e0", rangemode="tozero",
                 tickformat=",")
fig.update_yaxes(showgrid=True, gridcolor="#e0e0e0", row=1, col=2,
                 rangemode="tozero", tickformat=",")
fig.update_layout(
    title_text="Batch Scaling — QPS vs batch size  (★ = peak)",
    height=560,
    margin=dict(l=70, r=30, t=110, b=120),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(
        orientation="h",
        y=-0.28, x=0.5, xanchor="center",
        font=dict(size=11),
        itemwidth=30,
    ),
)
fig.show()